In [1]:
import pandas as pd

In [3]:
cd ..

/Users/anna_chen/Desktop/Recommand-System/MOST_committee


In [4]:
file_path = 'data/research_proj/115計算機學門審查/(勿對外公開資料或流傳)108-115年智慧計算學門大批專題計畫申請案件(含中英文摘要及關鍵字).xlsx'
statistic_folder_path = 'data/research_proj/115計算機學門審查/(密件)智慧計算學門統計1130130.xlsx'
apply_project_excel_file = pd.ExcelFile(file_path)
statistic_excel_file = pd.ExcelFile(statistic_folder_path)
years = ['108','109','110','111','112','113','114']

In [5]:
import re

result_df = {}
for year in years:
    apply_project_df = pd.read_excel(apply_project_excel_file, year)
    statistic_df = pd.read_excel(statistic_excel_file, f"{year}總計畫清單")
    
    # 將通過的計畫名稱處理成不含「子計畫x:」的格式
    pass_proj_name_list_original = statistic_df['計畫中文名稱'].to_list()
    
    # 創建兩個版本的通過清單：原始版本和處理過的版本
    pass_proj_name_list_processed = []
    if year == '113' or year == '114':
        for name in pass_proj_name_list_original:
            # 使用正則表達式移除「子計畫x:」格式
            if('子計畫' in name):
                processed_name = name.split('：')[-1].strip()
                pass_proj_name_list_processed.append(processed_name)
            else:
                pass_proj_name_list_processed.append(name)
    else:
        pass_proj_name_list_processed = pass_proj_name_list_original.copy()
    pass_list = []

    for i in range(len(apply_project_df)):
        if apply_project_df.iloc[i]['計畫中文名稱'] in pass_proj_name_list_original:
            pass_list.append('true')
        else:
            pass_list.append('false')
            
    apply_project_df['通過'] = pass_list  
    result_df[year] = apply_project_df


In [30]:
print(len(result_df))

7


In [31]:
all_project_file_with_pass = 'data/research_proj/115計算機學門審查/all_project.xlsx'

# 確保至少有一個工作表被創建
with pd.ExcelWriter(all_project_file_with_pass) as writer:
    sheets_created = False
    
    # 嘗試為每個年份創建工作表
    for year in years:
        try:
            if year in result_df:
                result_df[year].to_excel(writer, sheet_name=str(year), index=False)
                sheets_created = True
                print(f"已創建 {year} 年度的工作表")
            else:
                print(f"警告: result_df 中找不到 {year} 年度的資料")
        except Exception as e:
            print(f"處理 {year} 年度時出錯: {e}")
    
    # 如果沒有創建任何工作表，則創建一個空白工作表
    if not sheets_created:
        print("沒有找到任何年度的資料，創建一個空白工作表")
        pd.DataFrame().to_excel(writer, sheet_name="空白工作表", index=False)
print("所有年度的資料已成功寫入到 all_project.xlsx")

已創建 108 年度的工作表
已創建 109 年度的工作表
已創建 110 年度的工作表
已創建 111 年度的工作表
已創建 112 年度的工作表
已創建 113 年度的工作表
已創建 114 年度的工作表
所有年度的資料已成功寫入到 all_project.xlsx


In [32]:
pass_proj = {}
for year in years:
    pass_proj[year] = result_df[year][result_df[year]['通過'] == 'true']

In [33]:
pass_project_file_name = 'data/research_proj/115計算機學門審查/pass_project.xlsx'

# 確保至少有一個工作表被創建
with pd.ExcelWriter(pass_project_file_name) as writer:
    sheets_created = False
    
    # 嘗試為每個年份創建工作表
    for year in years:
        try:
            if year in pass_proj:
                pass_proj[year].to_excel(writer, sheet_name=str(year), index=False)
                sheets_created = True
                print(f"已創建 {year} 年度的工作表")
            else:
                print(f"警告: pass_proj 中找不到 {year} 年度的資料")
        except Exception as e:
            print(f"處理 {year} 年度時出錯: {e}")
    
    # 如果沒有創建任何工作表，則創建一個空白工作表
    if not sheets_created:
        print("沒有找到任何年度的資料，創建一個空白工作表")
        pd.DataFrame().to_excel(writer, sheet_name="空白工作表", index=False)
print("所有年度的資料已成功寫入到 all_project.xlsx")

已創建 108 年度的工作表
已創建 109 年度的工作表
已創建 110 年度的工作表
已創建 111 年度的工作表
已創建 112 年度的工作表
已創建 113 年度的工作表
已創建 114 年度的工作表
所有年度的資料已成功寫入到 all_project.xlsx


In [13]:
import pandas as pd
import os

years = ['108','109','110','111','112','113','114']

# 創建一個 Excel 寫入器
output_path = 'data/research_proj/115計算機學門審查/缺少的計畫原始資料.xlsx'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with pd.ExcelWriter(output_path) as writer:
    # 用於摘要的資料
    summary_data = []
    
    for year in years:
        print(f"\n===== {year}年度比較 =====")
        
        try:
            # 讀取兩個 Excel 檔案
            original_statistic_df = pd.read_excel('data/research_proj/115計算機學門審查/(密件)智慧計算學門統計1130130.xlsx', f"{year}總計畫清單")
            new_statistic_df = pd.read_excel('data/research_proj/115計算機學門審查/pass_project.xlsx', year)

            # 使用布林值 True 篩選通過的計畫 (在新清單中)
            new_pass_df = new_statistic_df[new_statistic_df['通過'] == True]   
            
            # 獲取計畫名稱列表
            original_pass_names = set(original_statistic_df['計畫中文名稱'].tolist())
            new_pass_names = set(new_pass_df['計畫中文名稱'].tolist())
            
            # 找出在原始清單中但不在新清單中的計畫
            missing_projects = original_pass_names - new_pass_names
            
            if missing_projects:
                print(f"\n找到 {len(missing_projects)} 個在原始清單中但不在新清單中的計畫")
                
                # 篩選出這些缺少的計畫在原始資料中的完整內容
                missing_df = original_statistic_df[original_statistic_df['計畫中文名稱'].isin(missing_projects)]
                
                # 將資料寫入 Excel 工作表
                missing_df.to_excel(writer, sheet_name=f"{year}年缺少計畫", index=False)
                
                # 添加到摘要資料
                summary_data.append({
                    '年度': f"{year}年",
                    '缺少計畫數量': len(missing_df),
                    '原始清單計畫總數': len(original_statistic_df),
                    '現有計劃數量': len(new_pass_df)
                })
            else:
                print(f"\n{year}年沒有發現缺少的計畫")
                summary_data.append({
                    '年度': f"{year}年",
                    '缺少計畫數量': 0,
                    '原始清單計畫總數': len(original_statistic_df),
                    '現有計劃數量': len(new_pass_df)
                })
                
        except Exception as e:
            print(f"處理 {year} 年度時出錯: {e}")
            summary_data.append({
                '年度': f"{year}年",
                '缺少計畫數量': '處理出錯',
                '原始清單計畫總數': '處理出錯'
            })
    
    # 創建摘要工作表
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name="缺少計畫摘要", index=False)

print(f"\n缺少的計畫原始資料已輸出至: {output_path}")



===== 108年度比較 =====

找到 127 個在原始清單中但不在新清單中的計畫

===== 109年度比較 =====

找到 112 個在原始清單中但不在新清單中的計畫

===== 110年度比較 =====

找到 159 個在原始清單中但不在新清單中的計畫

===== 111年度比較 =====

找到 170 個在原始清單中但不在新清單中的計畫

===== 112年度比較 =====

找到 170 個在原始清單中但不在新清單中的計畫

===== 113年度比較 =====

找到 209 個在原始清單中但不在新清單中的計畫

===== 114年度比較 =====

找到 41 個在原始清單中但不在新清單中的計畫

缺少的計畫原始資料已輸出至: data/research_proj/115計算機學門審查/缺少的計畫原始資料.xlsx
